In [2]:
import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration
from data_preprocessing import create_dataloader
from configs import CHECKPOINT, DATASET, BATCH_SIZE, MAX_LENGTH

/Users/zeinab/mambaforge/envs/llm_prefix_tuning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
t5_tokenizer = T5Tokenizer.from_pretrained(CHECKPOINT)
t5_model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT)

You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


## Data Preprocessing

In [4]:
# Load dataset
dataset = load_dataset(DATASET, "en-fr")['train']

# Shuffle the dataset
dataset = dataset.shuffle(seed=42)

# Split the dataset into train, valid, test (train_size=80%, valid_size=10%, test_size=10%)
train_valid_dataset, test_dataset = torch.utils.data.random_split(dataset, [0.9, 0.1])
train_dataset, valid_dataset = torch.utils.data.random_split(train_valid_dataset, [0.9, 0.1])

In [5]:
train_loader = create_dataloader(
    dataset=train_dataset, 
    tokenizer=t5_tokenizer, 
    shift_right_func=t5_model._shift_right,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

valid_loader = create_dataloader(
    dataset=valid_dataset, 
    tokenizer=t5_tokenizer,
    shift_right_func=t5_model._shift_right,
    batch_size=BATCH_SIZE,
    max_length=MAX_LENGTH
)

test_loader = create_dataloader(
    dataset=test_dataset, 
    tokenizer=t5_tokenizer, 
    shift_right_func=t5_model._shift_right, 
    batch_size=BATCH_SIZE, 
    max_length=MAX_LENGTH
)

In [6]:
print(f"Number of batches in train: {len(train_loader)}")
print(f"Number of batches in valid: {len(valid_loader)}")
print(f"Number of batches in test: {len(test_loader)}")

Number of batches in train: 6434
Number of batches in valid: 715
Number of batches in test: 795


In [10]:
# Get a sample batch
sample_batch = next(iter(train_loader))
    
print("\nSample batch shapes:")
for key, value in sample_batch.items():
    print(f"{key}: {value.shape}")

# Decode a sample from the batch
print("\n")
print("Input:", t5_tokenizer.decode(sample_batch['input_ids'][0].tolist(), skip_special_tokens=True))
print("Target:", t5_tokenizer.decode(sample_batch['labels'][0].tolist(), skip_special_tokens=True))


Sample batch shapes:
input_ids: torch.Size([16, 128])
attention_mask: torch.Size([16, 128])
labels: torch.Size([16, 128])
decoder_input_ids: torch.Size([16, 128])


Input: translate English to French: "What depth have we now reached?"
Target: --Et à quelle profondeur sommes-nous?
